In [2]:
import pandas as pd
import numpy as np
import os
import glob
import json
import uproot
import awkward as ak

In [ ]:
# === CONFIGURATION ===

BASE_PATH = "/home/aegis/ether/Research_HEP/Dataset_ver3/MC/reduce_root"

LUMI = 36100.0  # in pb^-1 (36.1 fb^-1)
REDUCTION = 0.6 # You processed 60% of total files

with open("mc_metadata.json", "r") as f:
    METADATA = json.load(f)


BRANCHES = [
    # --- Global Event & MET ---
    "MET_Core_AnalysisMETAuxDyn_mpx",
    "MET_Core_AnalysisMETAuxDyn_mpy",
    "MET_Core_AnalysisMETAuxDyn_sumet",
    "EventInfoAuxDyn_mcEventWeights",
    
    # --- Small-R Jets (Selection & Cleaning) ---
    "AnalysisJetsAuxDyn_pt",
    "AnalysisJetsAuxDyn_eta",
    "AnalysisJetsAuxDyn_phi",
    "AnalysisJetsAuxDyn_NNJvtPass",
    
    # --- Large-R Jets (AD Features - Expanded) ---
    "AnalysisLargeRJetsAuxDyn_pt",
    "AnalysisLargeRJetsAuxDyn_eta",
    "AnalysisLargeRJetsAuxDyn_phi",
    "AnalysisLargeRJetsAuxDyn_m",
    "AnalysisLargeRJetsAuxDyn_Tau1_wta",
    "AnalysisLargeRJetsAuxDyn_Tau2_wta",
    "AnalysisLargeRJetsAuxDyn_Tau3_wta",
    
    # --- Lepton & Tau Vetoes ---
    "AnalysisElectronsAuxDyn_DFCommonElectronsLHTight",
    "AnalysisMuonsAuxDyn_muonType",
    "AnalysisMuonsAuxDyn_quality",
    "AnalysisTauJetsAuxDyn_JetDeepSetTight",
    
    # --- Flavor Tagging ---
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu",
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc",
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"
]


: 

In [ ]:
def load_mc_events(split_type, branches, process="ttbar"):
    """
    split_type: 'train' (40%), 'val' (20%), 'ad' (20%), 'inference' (20%)
    branches: List of branch names to read from the ROOT files
    """
    # 1. Get and Sort Files Numerically
    ttbar_path = os.path.join(BASE_PATH, process)
    all_subprocesses = []

    # Iterate through: PhPy8EG_A14_ttbar_hdamp258p75_nonallhad, etc.
    for subp in METADATA[process].keys():
        subp_dir = os.path.join(ttbar_path, f"mc20_13TeV_MC_{subp}")
        print(f"Checking directory: {subp_dir}")
        # subp_path = os.path.join(ttbar_path, subp)
        # print(os.listdir(subp_path))
        if not os.path.exists(subp_dir):
            continue

        print(f"Processing subprocess: {subp}")
        
        # 1. Get and Sort Files Numerically
        files = sorted([os.path.join(subp_dir, f) for f in os.listdir(subp_dir) if f.endswith(".root")])
        n = len(files)
        try: 
            events = uproot.concatenate([f + ":CollectionTree" for f in files], branches)
        
            # Physics Weight Calculation: (Cross-Section * Lumi) / (Sum of Weights * 0.6)
            meta = METADATA[process][subp]
            norm = (meta['xsec_pb'] * LUMI) / (meta['sum_w'] * REDUCTION)
            
            # Generator weights are in a vector; usually we take index 0
            events["weight_phys"] = events["EventInfoAuxDyn_mcEventWeights"][:, 0] * norm
            all_subprocesses.append(events)
        except Exception as e:
            print(f"Error loading {subp}: {e}")
            continue
    # Combine nonallhad and allhad into one ttbar array
    return ak.concatenate(all_subprocesses)

ttbar_events_train = load_mc_events("train", branches=BRANCHES, process="ttbar")
diboson_events_train = load_mc_events("train", branches=BRANCHES, process="Diboson")
single_top_events_train = load_mc_events("train", branches=BRANCHES, process="Single_top")
multijet_events_train = load_mc_events("train", branches=BRANCHES, process="Multijet")
wjets_events_train = load_mc_events("train", branches=BRANCHES, process="Wjets")
zjets_events_train = load_mc_events("train", branches=BRANCHES, process="Zjets")

print("Data loading complete.")

Checking directory: /home/aegis/ether/Research_HEP/Dataset_ver3/MC/reduce_root/ttbar/mc20_13TeV_MC_PhPy8EG_A14_ttbar_hdamp258p75_nonallhad
Processing subprocess: PhPy8EG_A14_ttbar_hdamp258p75_nonallhad
Checking directory: /home/aegis/ether/Research_HEP/Dataset_ver3/MC/reduce_root/ttbar/mc20_13TeV_MC_PhPy8EG_A14_ttbar_hdamp258p75_allhad
Processing subprocess: PhPy8EG_A14_ttbar_hdamp258p75_allhad
Checking directory: /home/aegis/ether/Research_HEP/Dataset_ver3/MC/reduce_root/Diboson/mc20_13TeV_MC_Sh_2211_WlvWqq
Processing subprocess: Sh_2211_WlvWqq
Checking directory: /home/aegis/ether/Research_HEP/Dataset_ver3/MC/reduce_root/Diboson/mc20_13TeV_MC_Sh_2211_WlvZqq
Processing subprocess: Sh_2211_WlvZqq
Checking directory: /home/aegis/ether/Research_HEP/Dataset_ver3/MC/reduce_root/Diboson/mc20_13TeV_MC_Sh_2211_WqqZvv
Processing subprocess: Sh_2211_WqqZvv
Checking directory: /home/aegis/ether/Research_HEP/Dataset_ver3/MC/reduce_root/Diboson/mc20_13TeV_MC_Sh_2211_ZqqZvv
Processing subprocess: S

In [ ]:
def calc_dr_matrix(eta1, phi1, eta2, phi2):
        # Create a grid to compare every object in collection 1 to every object in collection 2
        deta = eta1[:, :, np.newaxis] - eta2[:, np.newaxis, :]
        dphi = np.abs(phi1[:, :, np.newaxis] - phi2[:, np.newaxis, :])
        dphi = ak.where(dphi > np.pi, 2*np.pi - dphi, dphi)
        return np.sqrt(deta**2 + dphi**2)

def lepton_selection(events):
    """
    ATLAS Paper Lepton Selection 
    """
    w_initial = ak.sum(events["weight_phys"])

    # --- 1. Electron Selection ---
    # ele_pt = events["AnalysisElectronsAuxDyn_pt"] / 1000.0
    # ele_eta = events["AnalysisElectronsAuxDyn_eta"]
    # DFCommonElectronsLHTight is a boolean for the "Likelihood Tight" ID
    ele_id = events["AnalysisElectronsAuxDyn_DFCommonElectronsLHTight"] == 1
    # Standard cuts: pT > 7 GeV, |eta| < 2.5 (excluding crack region 1.37-1.52)
    # ele_kin_mask = (ele_pt > 7) & (abs(ele_eta) < 2.5) & ((abs(ele_eta) < 1.37) | (abs(ele_eta) > 1.52))
    good_electrons = ele_id

    # --- 2. Muon Selection ---
    # mu_pt = events["AnalysisMuonsAuxDyn_pt"] / 1000.0
    # mu_eta = events["AnalysisMuonsAuxDyn_eta"]
    
    # Selection criteria based on professor's verification:
    # Quality 8 or 9 for Medium
    mu_quality = (events["AnalysisMuonsAuxDyn_quality"] == 8) | \
                 (events["AnalysisMuonsAuxDyn_quality"] == 9)
    
    # Combined Muon Type (Type == 0)
    mu_type = events["AnalysisMuonsAuxDyn_muonType"] == 0
    
    # Kinematics: pT > 7 GeV, |eta| < 2.5
    # mu_kin = (mu_pt > 7) & (abs(mu_eta) < 2.5)
    
    good_muons = mu_quality & mu_type

    # --- 3. Tau Selection (Veto Candidates) ---
    # tau_pt = events["AnalysisTauJetsAuxDyn_pt"] / 1000.0
    # tau_eta = events["AnalysisTauJetsAuxDyn_eta"]
    # RNN identification: JetDeepSetTight
    tau_id = events["AnalysisTauJetsAuxDyn_JetDeepSetTight"] == 1
    
    # Standard Tau acceptance: pT > 20 GeV, |eta| < 2.5
    # tau_kin = (tau_pt > 20) & (abs(tau_eta) < 2.5)
    good_taus = tau_id

    return good_electrons, good_muons, good_taus

def event_cleaning(events):
    """ 
    ATLAS Paper Selection 
    1. Events are required to have at least two jets within |𝜂| < 2.8, the leading jet is required to have pT > 250 GeV
    2. Requirements on the jet-vertex tagger discriminant are applied to jets with pT below 60 GeV
    3. At least one jet within Δϕ = 2.0 of the MET direction.
    4. b-jet veto: Events with two or more b-tagged jets (77% WP) are vetoed.
    6. Lepton Selection and Tau Veto
    8. MET Calculation and Selection
    9. Events are required to have at least two large jets
    """
    # --- 1. Jet Kinematics Selection ---
    w_initial = ak.sum(events["weight_phys"])

    jet_pt = events["AnalysisJetsAuxDyn_pt"] / 1000.0
    jet_eta = events["AnalysisJetsAuxDyn_eta"]

    # Only look at jets in the tracker volume
    in_acceptance = abs(jet_eta) < 2.8
    accepted_pts = jet_pt[in_acceptance]

    # Requirement: At least 2 jets in acceptance
    has_two_jets = ak.num(accepted_pts, axis=1) >= 2

    # Requirement: Leading jet > 250 GeV
    # Requirement: Sub-leading (index 1) jet > 30 GeV
    leading_pt = ak.pad_none(accepted_pts, 2, axis=1)[:, 0]
    subleading_pt = ak.pad_none(accepted_pts, 2, axis=1)[:, 1]
    
    pass_pts = (ak.fill_none(leading_pt > 250, False)) & (ak.fill_none(subleading_pt > 30, False))

    final_mask_jet = has_two_jets & pass_pts

    # Weight Stats
    w_after_jets = ak.sum(events[final_mask_jet]["weight_phys"])
    print(f"Jet Selection: {ak.sum(final_mask_jet)}/{len(events)} passed ({100*ak.sum(final_mask_jet)/len(events):.2f}%)")
    print(f"Jet Weighted Efficiency: {100 * w_after_jets / w_initial:.2f}%")
    
    events = events[final_mask_jet]
    # --- 3. JVT Cleaning ---
    w_initial = ak.sum(events["weight_phys"])
    jvt_pass = events["AnalysisJetsAuxDyn_NNJvtPass"]
    jet_pt = events["AnalysisJetsAuxDyn_pt"] / 1000.0
    jet_eta = events["AnalysisJetsAuxDyn_eta"]

    # Apply JVT only to jets with pT < 60 GeV and |eta| < 2.4
    low_pt_mask = (jet_pt < 60) & (abs(jet_eta) < 2.4)
    jvt_mask = ak.all(ak.where(low_pt_mask, jvt_pass, True), axis=1)    
    
    w_after_jvt = ak.sum(events[jvt_mask]["weight_phys"])
    print(f"JVT Selection: {ak.sum(jvt_mask)}/{len(events)} events passed ({100*ak.sum(jvt_mask)/len(events):.2f}%)")
    print(f"JVT Weighted Efficiency: {100 * w_after_jvt / w_initial:.2f}%")

    events = events[jvt_mask]

    # --- 4. MET-Jet Delta Phi Selection ---
    w_initial = ak.sum(events["weight_phys"])

    # Calculate MET Phi from components
    met_px = events["MET_Core_AnalysisMETAuxDyn_mpx"][:, 0]
    met_py = events["MET_Core_AnalysisMETAuxDyn_mpy"][:, 0]
    met_phi = np.arctan2(met_py, met_px)

    jet_phi = events["AnalysisJetsAuxDyn_phi"]
    # Calculate Delta Phi and wrap it to [-pi, pi]
    dphi = abs(jet_phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)

    # Requirement: At least one jet with dphi < 2.0
    # Note: We usually apply this to all jets or just jets in acceptance? 
    # Standard practice is all "Signal" jets (the ones in AnalysisJets)
    pass_dphi_mask = ak.any(dphi < 2.0, axis=1)
    
    w_after_dphi = ak.sum(events[pass_dphi_mask]["weight_phys"])
    print(f"MET-Jet dPhi Selection: {ak.sum(pass_dphi_mask)}/{len(events)} events passed ({100*ak.sum(pass_dphi_mask)/len(events):.2f}%)")
    print(f"MET-Jet dPhi Weighted Efficiency: {100 * w_after_dphi / w_initial:.2f}%")
    
    events = events[pass_dphi_mask]

    # --- 5. B-Jet Veto ---
    w_initial = ak.sum(events["weight_phys"])

    # 1. Extract probabilities
    pb = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"]
    pc = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc"]
    pu = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu"]
    

    # 3. Calculate DL1dv01 Discriminant
    # Small epsilon (1e-10) prevents division by zero
    fc = 0.080
    dl1_score = np.log(pb / (fc * pc + (1 - fc) * pu + 1e-10))

    # 4. Define the 77% Working Point threshold
    # Note: 2.45 is the standard threshold for DL1dv01 @ 77% WP
    is_bjet = dl1_score > 1.42 

    # 5. Requirement: Events with TWO OR MORE b-jets are VETOED
    # This means we keep events where num_bjets < 2
    num_bjets = ak.sum(is_bjet, axis=1)
    bveto_mask = (num_bjets < 2)

    # Weight Stats
    w_after_bveto = ak.sum(events[bveto_mask]["weight_phys"])
    print(f"B-Jet Veto: {ak.sum(bveto_mask)}/{len(events)} events passed ({100*ak.sum(bveto_mask)/len(events):.2f}%)")
    print(f"B-Veto Weighted Efficiency: {100 * w_after_bveto / w_initial:.2f}%")

    events = events[bveto_mask]
    # --- 7. Lepton Selection and Tau Veto ---
    w_initial = ak.sum(events["weight_phys"])
    e_mask, m_mask, tau_mask = lepton_selection(events)
    n_tau = ak.sum(tau_mask, axis=1)

    lepton_mask = (n_tau == 0)

    w_after_lepton = ak.sum(events[lepton_mask]["weight_phys"])
    print(f"Lepton Selection & Tau Veto: {ak.sum(lepton_mask)}/{len(events)} events passed ({100*ak.sum(lepton_mask)/len(events):.2f}%)")
    print(f"Lepton & Tau Weighted Efficiency: {100 * w_after_lepton / w_initial:.2f}%") 

    events = events[lepton_mask]

    events["final_jet_mask"] = final_mask_jet
    events["final_ele_mask"] = e_mask
    events["final_mu_mask"] = m_mask
    # 9. MET Calculation and Selection
    w_initial = ak.sum(events["weight_phys"])
    # 1. Get the Soft Term (Assuming Index 0 is PVSoftTrk or SoftClus)
    soft_px = (events["MET_Core_AnalysisMETAuxDyn_mpx"][:, 0])/1000
    soft_py = (events["MET_Core_AnalysisMETAuxDyn_mpy"][:, 0])/1000

    # 2. Sum the PX/PY of CLEANED Jets (Step 8 jet_mask)
    # Note: Using the original AnalysisJets branches with the mask
    all_j_px = (events["AnalysisJetsAuxDyn_pt"] * np.cos(events["AnalysisJetsAuxDyn_phi"])) / 1000.0
    all_j_py = (events["AnalysisJetsAuxDyn_pt"] * np.sin(events["AnalysisJetsAuxDyn_phi"])) / 1000.0
    sum_j_px = ak.sum(all_j_px[final_mask_jet], axis=1)
    sum_j_py = ak.sum(all_j_py[final_mask_jet], axis=1)

    # # 3. Sum the PX/PY of CLEANED Electrons (Step 8 e_mask)
    # all_e_px = (events["AnalysisElectronsAuxDyn_pt"] * np.cos(events["AnalysisElectronsAuxDyn_phi"])) / 1000.0
    # all_e_py = (events["AnalysisElectronsAuxDyn_pt"] * np.sin(events["AnalysisElectronsAuxDyn_phi"])) / 1000.0
    # sum_e_px = ak.sum(all_e_px[e_mask], axis=1)
    # sum_e_py = ak.sum(all_e_py[e_mask], axis=1)

    # 4. Final Reconstruction (Negative Vector Sum)
    # We EXCLUDE Muons here to satisfy the "muons are invisible" requirement.
    # If we wanted standard MET, we would subtract sum_mu_px as well.
    met_recalc_px = -(sum_j_px + soft_px)
    met_recalc_py = -(sum_j_py + soft_py)

    met_recalc = np.sqrt(met_recalc_px**2 + met_recalc_py**2)
    events["met_recalc_pt"] = np.sqrt(met_recalc_px**2 + met_recalc_py**2)
    events["met_recalc_phi"] = np.arctan2(met_recalc_py, met_recalc_px)

    
    # 5. MET Selection: MET > 250 GeV
    met_mask = met_recalc > 250

    w_after_met = ak.sum(events[met_mask]["weight_phys"])
    print(f"MET Selection: {ak.sum(met_mask)}/{len(events)} events passed ({100*ak.sum(met_mask)/len(events):.2f}%)")
    print(f"MET Weighted Efficiency: {100 * w_after_met / w_initial:.2f}%") 

    events = events[met_mask]
    # 10. Require at least two large-R jets
    w_initial = ak.sum(events["weight_phys"])
    large_jet_pt = events["AnalysisLargeRJetsAuxDyn_pt"] / 1000.0
    n_large_jets = ak.num(large_jet_pt, axis=1)
    large_jet_mask = (n_large_jets >= 2)

    w_after_largejet = ak.sum(events[large_jet_mask]["weight_phys"])
    print(f"Large-R Jet Selection: {ak.sum(large_jet_mask)}/{len(events)} events passed ({100*ak.sum(large_jet_mask)/len(events):.2f}%)")
    print(f"Large-R Jet Weighted Efficiency: {100 * w_after_largejet / w_initial:.2f}%")

    events = events[large_jet_mask]

    
    return events

ttbar_preselection = event_cleaning(ttbar_events_train)
diboson_preselection = event_cleaning(diboson_events_train)
single_top_preselection = event_cleaning(single_top_events_train)   
qcd_preselection = event_cleaning(multijet_events_train)
wjets_preselection = event_cleaning(wjets_events_train)
zjets_preselection = event_cleaning(zjets_events_train)

In [ ]:
def region_selection(events, region):
    # --- 1. Calculate Required Variables ---
    
    # HT: Scalar sum of pT of CLEANED jets
    # Using the final_jet_mask from your preselection
    jet_pt = events["AnalysisJetsAuxDyn_pt"] / 1000.0
    clean_jets_pt = jet_pt[events["final_jet_mask"]]
    ht = ak.sum(clean_jets_pt, axis=1)

    # MET: Using your recalculated MET from preselection
    met = events["met_recalc_pt"]

    # Lepton Counts (using masks from preselection)
    n_ele = ak.sum(events["final_ele_mask"], axis=1)
    n_mu = ak.sum(events["final_mu_mask"], axis=1)
    
    # B-jet Count (using the same logic as your B-Jet Veto step)
    # Re-calculating b-jets based on the 77% WP (score > 1.42)
    pb = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"]
    pc = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc"]
    pu = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu"]
    fc = 0.080
    dl1_score = np.log(pb / (fc * pc + (1 - fc) * pu + 1e-10))
    is_bjet = dl1_score > 1.42
    n_bjets = ak.sum(is_bjet, axis=1)

    # Common Kinematic Cuts for SR and all CRs
    kin_mask = (met > 600) & (ht > 600)

    # --- 2. Define Regions ---

    if region == "SR":
        # No electrons, no muons, and at most one b-jet
        sr_mask = kin_mask & (n_ele == 0) & (n_mu == 0) & (n_bjets <= 1)
        return events[sr_mask]

    elif region == "1L":
        # Exactly 1 muon, no b-jets
        cr1l_mask = kin_mask & (n_mu == 1) & (n_ele == 0) & (n_bjets == 0)
        return events[cr1l_mask]

    elif region == "1L1B":
        # Exactly 1 muon, exactly 1 b-jet
        cr1l1b_mask = kin_mask & (n_mu == 1) & (n_ele == 0) & (n_bjets == 1)
        return events[cr1l1b_mask]

    elif region == "2L":
        # Exactly 2 muons, no b-jets
        mu_mask = events["final_mu_mask"]
        two_mu_mask = (n_mu == 2) & (n_ele == 0) & (n_bjets == 0)
        
        # Invariant Mass check for the two muons
        # We need to filter events first to avoid indexing issues
        events_2mu = events[two_mu_mask]
        
        mu_pt = (events_2mu["AnalysisMuonsAuxDyn_pt"] / 1000.0)[events_2mu["final_mu_mask"]]
        mu_eta = events_2mu["AnalysisMuonsAuxDyn_eta"][events_2mu["final_mu_mask"]]
        mu_phi = events_2mu["AnalysisMuonsAuxDyn_phi"][events_2mu["final_mu_mask"]]
        mu_charge = events_2mu["AnalysisMuonsAuxDyn_charge"][events_2mu["final_mu_mask"]]

        # Calculate m_ll = sqrt(2 * pt1 * pt2 * (cosh(eta1-eta2) - cos(phi1-phi2)))
        # Using index 0 and 1 for the two muons
        m_ll = np.sqrt(2 * mu_pt[:, 0] * mu_pt[:, 1] * (
            np.cosh(mu_eta[:, 0] - mu_eta[:, 1]) - np.cos(mu_phi[:, 0] - mu_phi[:, 1])
        ))
        
        # Check for opposite charge and mass window [66, 116]
        opp_charge = mu_charge[:, 0] != mu_charge[:, 1]
        mass_window = (m_ll >= 66) & (m_ll <= 116)
        
        final_2l_events = events_2mu[opp_charge & mass_window]
        # Further filter by the HT/MET kinematic mask
        return final_2l_events[(final_2l_events["met_recalc_pt"] > 600) & 
                               (ak.sum(final_2l_events["AnalysisJetsAuxDyn_pt"][final_2l_events["final_jet_mask"]], axis=1)/1000.0 > 600)]

    else:
        raise ValueError("Invalid region. Must be 'SR', '1L', '1L1B', or '2L'.")
    

ttbar_SR = region_selection(ttbar_preselection, "SR")
diboson_SR = region_selection(diboson_preselection, "SR")
single_top_SR = region_selection(single_top_preselection, "SR")
qcd_SR = region_selection(qcd_preselection, "SR")
wjets_SR = region_selection(wjets_preselection, "SR")
zjets_SR = region_selection(zjets_preselection, "SR")
print("Region selection complete.")

--- Processing MC Files ---
Loading Diboson...
Processed MC_Diboson: SR=243, CR_Train=44531, CR_Val=22266
Loading Multijet...
Processed MC_Multijet: SR=166150, CR_Train=22415042, CR_Val=11207522
Loading Wjets...
Processed MC_Wjets: SR=44767, CR_Train=7171832, CR_Val=3585916
Loading Zjets...
Processed MC_Zjets: SR=3129, CR_Train=511916, CR_Val=255959
Loading ttbar...
Processed MC_ttbar: SR=753, CR_Train=111890, CR_Val=55946
Loading Single_top...
Processed MC_Single_top: SR=31, CR_Train=4426, CR_Val=2214

--- Processing Data Files ---
Loading PeriodA...
Processed Data_PeriodA: SR=520, CR_Train=5667894, CR_Val=2833948
Loading PeriodB...
Processed Data_PeriodB: SR=5418, CR_Train=14637318, CR_Val=7318660
Loading PeriodC...
Processed Data_PeriodC: SR=12984, CR_Train=23987341, CR_Val=11993671
Loading PeriodD...
Processed Data_PeriodD: SR=37107, CR_Train=30918520, CR_Val=15459260
Loading PeriodE...
Processed Data_PeriodE: SR=6479, CR_Train=9341111, CR_Val=4670556
Loading PeriodF...
Processed D